# Tuning the FGW trade-off parameter $\alpha$

$\alpha \in [0, 1]$ balances the two terms of the Fused Gromov-Wasserstein distance between sentence graphs:

* $(1-\alpha)$ × **feature cost**: squared Euclidean distance between cue-description embeddings (normalized by its maximum, so in $[0, 1]$);
* $\alpha$ × **structure cost**: squared difference between temporal gaps $C(i,j)$, in words².

The two costs are on very different scales: gaps reach tens of words while the feature cost is at most 1. With raw gaps the structure term dominates and the mean distance grows almost linearly with $\alpha$, so the optimum says more about units than about the data. With `NORMALIZE_STRUCTURE = True`, $C$ is divided by the largest gap found in the corpus, so both costs lie in $[0, 1]$. **The same scaling must then be used in every analysis that uses the selected $\alpha$.**

For each seed and each dimension (ability, benevolence), we draw a stratified sample of sentences (same number per **instructed** level), compute all pairwise FGW distances for a grid of $\alpha$, and score each $\alpha$ with:

| criterion | definition |
|---|---|
| `mean_distance` | mean pairwise distance (criterion described in the paper) |
| `ratio` | mean between-level distance / mean within-level distance |
| `silhouette` | silhouette score of the instructed levels on the precomputed distances |
| `accuracy` | leave-one-out accuracy of the "nearest level" rule (each graph gets the level whose other graphs are closest on average) |

Only instructed levels are used, never perceived levels, so the choice of $\alpha$ does not use the data evaluated in H2.

In [ ]:
import os
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import ot
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score

sys.path.append(os.path.realpath("../lib"))
from graph_multimodalsentence import Graph, load_gesture_table, parse_cues, temporal_gap
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
DATA_DIR = "../data_with_text_tag"
OUTPUT_DIR = "alpha_tuning"

DIMENSIONS = ["ability", "benevolence"]
GENDERS = ["neutral", "man", "woman"]
MODELS = ["gemma", "gpt", "mistral"]
TASKS = {"": "navigation", "_QA": "QA"}
LEVELS = ["Low", "Medium", "High"]

SEEDS = list(range(20))               # one stratified sample per seed and dimension
N_PER_LEVEL = 10                      # sentences per level -> 30 graphs, 435 pairs per sample
ALPHAS = np.round(np.linspace(0, 1, 51), 2)
CRITERION = "ratio"                   # "mean_distance", "ratio", "silhouette" or "accuracy"
NORMALIZE_STRUCTURE = True            # divide C by the largest gap of the corpus

# Sentences used as stimuli in the perceptual study, excluded so that alpha is
# chosen on independent data (fill with their generated_id if available).
STUDY_IDS = set()

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data

All generated sentences of the `full_*` conditions, with their instructed level. The `controle` files have no instructed level and the `*_ratedbyFGW` files are derived outputs, so both are left out.

In [ ]:
def load_condition(explanation, dimension, gender, model, task_suffix):
    path = os.path.join(DATA_DIR, "generated_instructions_with_intention_explanation_"
                        f"{explanation}_gender_{gender}_{model}{task_suffix}.csv")
    if not os.path.exists(path):
        return None
    data = pd.read_csv(path)
    return pd.DataFrame({
        "generated_id": data["generated_id"],
        "sentence": data["instruction_with_intention"],
        "dimension": dimension,
        "level": data[dimension].astype(str).str.split(" ").str[0],
        "gender": gender,
        "model": model,
        "task": TASKS[task_suffix],
    })


frames = []
for dimension in DIMENSIONS:
    for explanation in [f"full_{dimension}", "full_ability_benevolence"]:
        for gender in GENDERS:
            for model in MODELS:
                for task_suffix in TASKS:
                    frame = load_condition(explanation, dimension, gender, model, task_suffix)
                    if frame is not None:
                        frames.append(frame)

df = pd.concat(frames, ignore_index=True).dropna(subset=["sentence"])
df = df[df["level"].isin(LEVELS) & ~df["generated_id"].isin(STUDY_IDS)]
df.groupby(["dimension", "level"]).size().unstack()

In [ ]:
def max_gap(sentence, gesture_table=load_gesture_table()):
    cues, _ = parse_cues(sentence, gesture_table)
    return max((temporal_gap(a, b) for a in cues for b in cues), default=0)


if NORMALIZE_STRUCTURE:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # unknown gestures are reported when graphs are built
        STRUCTURE_SCALE = df["sentence"].map(max_gap).max()
else:
    STRUCTURE_SCALE = 1
print(f"Structure scale: {STRUCTURE_SCALE} words")

## Graphs and distances

Embeddings are cached by description, since the same cue descriptions occur in many sentences. For a pair of graphs, the feature cost $M$ and the structure matrices $C$ do not depend on $\alpha$: they are computed once and only the FGW optimization is repeated for each $\alpha$.

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))


def fgw_distances(g1, g2, alphas):
    # FGW distance between two graphs for every alpha, with uniform node weights.
    # The feature cost is computed as in Fused_Gromov_Wasserstein_distance.graph_d.
    M = ot.dist(g1.all_matrix_attr(), g2.all_matrix_attr(), metric="sqeuclidean")
    if M.max() != 0:
        M = M / M.max()
    C1 = g1.distance_matrix() / STRUCTURE_SCALE
    C2 = g2.distance_matrix() / STRUCTURE_SCALE
    p = np.ones(len(C1)) / len(C1)
    q = np.ones(len(C2)) / len(C2)
    fgw = Fused_Gromov_Wasserstein_distance(features_metric="sqeuclidean", method="temporal")
    distances = []
    for alpha in alphas:
        fgw.alpha = alpha
        _, log = fgw.calc_fgw(M, C1, C2, p, q)
        distances.append(log["loss"][-1])
    return np.array(distances)


def pairwise_distances(graphs, alphas):
    # Distance matrices for all alphas, shape (len(alphas), n, n). Failures are NaN.
    n = len(graphs)
    D = np.zeros((len(alphas), n, n))
    n_fail = 0
    for i in range(n):
        for j in range(i + 1, n):
            try:
                d = fgw_distances(graphs[i], graphs[j], alphas)
            except Exception:
                d = np.nan
                n_fail += 1
            D[:, i, j] = D[:, j, i] = d
    if n_fail:
        print(f"  {n_fail} failed pairs (stored as NaN)")
    return D

## Scores

In [ ]:
def scores(D, labels):
    # Mean distance, separation ratio, silhouette and LOO nearest-level accuracy.
    labels = np.asarray(labels)
    same = labels[:, None] == labels[None, :]
    off_diag = ~np.eye(len(labels), dtype=bool)
    valid = ~np.isnan(D)

    mean_distance = D[off_diag & valid].mean()
    within = D[same & off_diag & valid].mean()
    between = D[~same & valid].mean()
    ratio = between / within if within > 0 else np.nan

    keep = valid.all(axis=1)  # silhouette needs a complete matrix
    if len(np.unique(labels[keep])) == len(LEVELS):
        silhouette = silhouette_score(D[np.ix_(keep, keep)], labels[keep], metric="precomputed")
    else:
        silhouette = np.nan

    correct = []
    for i in range(len(labels)):
        mean_to_level = {level: D[i, (labels == level) & off_diag[i] & valid[i]].mean()
                         for level in LEVELS}
        correct.append(min(mean_to_level, key=mean_to_level.get) == labels[i])

    return {"mean_distance": mean_distance, "ratio": ratio,
            "silhouette": silhouette, "accuracy": np.mean(correct)}

## Run

Distance matrices are saved per dimension and seed, and reloaded if they already exist with the same $\alpha$ grid and structure scale.

In [ ]:
records = []
for dimension in DIMENSIONS:
    pool = df[df["dimension"] == dimension]
    for seed in SEEDS:
        sample = pd.concat([pool[pool["level"] == level].sample(N_PER_LEVEL, random_state=seed)
                            for level in LEVELS])
        labels = sample["level"].to_numpy()

        path = os.path.join(OUTPUT_DIR, f"fgw_distances_{dimension}_seed{seed}.npz")
        cached = np.load(path) if os.path.exists(path) else None
        if (cached is not None and np.array_equal(cached["alphas"], ALPHAS)
                and cached["scale"] == STRUCTURE_SCALE):
            D = cached["D"]
        else:
            start = time.time()
            graphs = [Graph.from_sentence(s, embedding) for s in sample["sentence"]]
            D = pairwise_distances(graphs, ALPHAS)
            np.savez(path, D=D, alphas=ALPHAS, scale=STRUCTURE_SCALE, generated_id=sample["generated_id"].to_numpy())
            print(f"{dimension}, seed {seed}: {time.time() - start:.0f} s")

        for a, alpha in enumerate(ALPHAS):
            records.append({"dimension": dimension, "seed": seed, "alpha": alpha,
                            **scores(D[a], labels)})

results = pd.DataFrame(records)
results.to_csv(os.path.join(OUTPUT_DIR, "alpha_selection_results.csv"), index=False)

## Choice of $\alpha$

A single $\alpha$ is used for both dimensions: the one maximizing the criterion averaged over seeds and dimensions.

In [ ]:
CRITERIA = ["mean_distance", "ratio", "silhouette", "accuracy"]
summary = results.groupby(["dimension", "alpha"])[CRITERIA].mean()
overall = results.groupby("alpha")[CRITERIA].mean()
best_alpha = overall[CRITERION].idxmax()

best = pd.DataFrame({criterion: overall[criterion].idxmax() for criterion in CRITERIA}, index=["best alpha"])
print(f"Selected alpha ({CRITERION}): {best_alpha}")
print(overall.loc[best_alpha])
best

In [ ]:
labels = {"mean_distance": "Mean pairwise FGW distance",
          "ratio": "Between / within distance",
          "silhouette": "Silhouette score",
          "accuracy": "LOO nearest-level accuracy"}

fig, axes = plt.subplots(1, len(CRITERIA), figsize=(4.5 * len(CRITERIA), 4), sharex=True)
for ax, criterion in zip(axes, CRITERIA):
    sns.lineplot(data=results, x="alpha", y=criterion, hue="dimension", errorbar=("ci", 95), ax=ax)
    if criterion == "accuracy":
        ax.axhline(1 / len(LEVELS), ls="--", c="grey", lw=1, label="chance")
    ax.axvline(best_alpha, ls=":", c="k", lw=1)
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(labels[criterion])
    ax.legend()
fig.suptitle(rf"FGW distance as a function of $\alpha$ (mean and 95% CI over {len(SEEDS)} seeds); "
             rf"selected $\alpha$ = {best_alpha} ({CRITERION})")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "tunealpha.png"), dpi=300)
plt.show()